# Лабораторна робота 4. Розпізнавання об'єктів на зображеннях з набору даних CIFAR-10

Згорткова нейронна мережа (Keras). Виконав: Бабин Богдан Олегович, група 443-4

**Перед запуском:** Среда выполнения → Сменить среду выполнения → **T4 GPU**, потім «Выполнить все».
На комірці завантаження фото натисніть «Выбрать файлы» і оберіть свої зображення.

> **Оновлення коду до нового API Keras 3** (базовий ноутбук написаний під старий Keras):
> - `keras.preprocessing.image` → `keras.utils.load_img / img_to_array`;
> - `model.save_weights("cifar10_model1.h5")` у Keras 3 видає помилку — ім'я файлу ваг має закінчуватися на `.weights.h5`, тому `cifar10_model.weights.h5`.

In [ ]:
import numpy as np
import pandas as pd
import os, shutil, time
from keras.datasets import cifar10
from keras.models import Sequential, model_from_json
from keras.layers import Dense, Flatten, Input
from keras.layers import Dropout, BatchNormalization
from keras.layers import Conv2D, MaxPooling2D
from keras.callbacks import EarlyStopping
from keras import utils
from keras.utils import load_img, img_to_array
from google.colab import files
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
%matplotlib inline

utils.set_random_seed(42)

In [ ]:
# Розмір міні-вибірки
batch_size = 128
# Кількість класів зображень
nb_classes = 10
# Кількість епох для навчання
nb_epoch = 25
# Розмір зображеннь
img_rows, img_cols = 32, 32
# Кількість каналів в зображенні
img_channels = 3
# Назви класівз набору даних  CIFAR-10
classes=['літак', 'автомобіль', 'птах', 'кіт', 'олень', 'собака', 'жаба', 'кінь', 'корабель', 'вантажівка']

## 1–2. Підготовка даних

**Завантажуємо дані**

In [ ]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

**Перегляд прикладів даних**

In [ ]:
n = 1
plt.imshow(X_train[n])
plt.show()
print("Номер класу:", y_train[n])
print("Тип об'єкту:", classes[y_train[n][0]])

In [ ]:
# Перші 20 зображень навчального набору
plt.figure(figsize=(12, 5))
for i in range(20):
    plt.subplot(2, 10, i + 1)
    plt.imshow(X_train[i]); plt.axis('off')
    plt.title(classes[y_train[i][0]], fontsize=9)
plt.show()

**Нормалізуємо  дані**

In [ ]:
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
X_train /= 255
X_test /= 255

**Перетворюємо вірні відповіді в формат one hot encoding**

In [ ]:
Y_train = utils.to_categorical(y_train, nb_classes)
Y_test = utils.to_categorical(y_test, nb_classes)

## 3. Створюємо нейронну мережу (базова версія)

In [ ]:
# Створюємо послідовну модель
model = Sequential()
# Перший згортковий шар
model.add(Conv2D(32, (3, 3), padding='same',
                        input_shape=(32, 32, 3), activation='relu'))

# Другий згортковий шар
model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
# Перший шар підвибірки
model.add(MaxPooling2D(pool_size=(2, 2)))
# Шар регуляризації Dropout
model.add(Dropout(0.25))

# Третій згортковий шар
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
# Четвертий згортковий шар
model.add(Conv2D(64, (3, 3), activation='relu'))
# Другий шар підвибірки
model.add(MaxPooling2D(pool_size=(2, 2)))
# Шар регуляризації Dropout
model.add(Dropout(0.25))
# Шар перетворення даних з 2D представлення в пласке
model.add(Flatten())
# Повнозв'язаний шар для класифікації
model.add(Dense(512, activation='relu'))
# Шар регуляризації Dropout
model.add(Dropout(0.5))
# Вихідний Повнозв'язаний шар
model.add(Dense(nb_classes, activation='softmax'))

**Друкуємо інформацію щодо мережі**

In [ ]:
model.summary()

## 4. Компілюємо та навчаємо модель

In [ ]:
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
start = time.time()
history = model.fit(X_train, Y_train,
              batch_size=batch_size,
              epochs=nb_epoch,
              validation_split=0.1,
              shuffle=True,
              verbose=2)
base_time = time.time() - start

## 5. Оцінюємо якість  навчання мережі

In [ ]:
# Оцінюємо якість навчання моделі на тестових даних
scores = model.evaluate(X_test, Y_test, verbose=0)
print("Точність роботи на тестових даних: %.2f%%" % (scores[1]*100))

## 6. Візуалізація історії навчання: точність і втрати

In [ ]:
def plot_history(history, title=''):
    h = history.history
    epochs = range(1, len(h['accuracy']) + 1)
    plt.figure(figsize=(13, 4))
    # Графік точності
    plt.subplot(1, 2, 1)
    plt.plot(epochs, h['accuracy'], 'bo-', markersize=3, label='Training acc')
    plt.plot(epochs, h['val_accuracy'], 'r', label='Validation acc')
    plt.xlabel('Epochs'); plt.ylabel('Accuracy'); plt.title('Точність'); plt.legend(); plt.grid(alpha=.3)
    # Графік втрат
    plt.subplot(1, 2, 2)
    plt.plot(epochs, h['loss'], 'bo-', markersize=3, label='Training loss')
    plt.plot(epochs, h['val_loss'], 'r', label='Validation loss')
    plt.xlabel('Epochs'); plt.ylabel('Loss'); plt.title('Втрати'); plt.legend(); plt.grid(alpha=.3)
    plt.suptitle(title); plt.tight_layout(); plt.show()

plot_history(history, 'Базова модель')

**Аналіз графіків базової моделі** *(заповніть після запуску)*: чи ще зростає точність на перевірці, чи вийшла на плато,
чи розходяться лінії втрат (ознака перенавчання), з якої епохи.

## 7. Ручний підбір гіперпараметрів

Кожен експеримент змінює **один** параметр відносно базової моделі (пункти 1–8 завдання), останній — комбінує зміни.
Щоб модель не перенавчалась, у кожному експерименті використано `EarlyStopping`: навчання зупиняється,
якщо втрати на перевірці не зменшуються 5 епох, і відновлюються ваги з найкращої епохи.

> Усі експерименти разом займуть ~30–40 хвилин на T4.

In [ ]:
def build_model(blocks=(32, 64), kernel_first=3, kernel=3, dense=(512,),
                conv_dropout=0.25, dense_dropout=0.5, batchnorm=False, optimizer='adam'):
    """blocks — кількість карт ознак у кожному згортковому блоці (2 шари Conv2D + MaxPooling + Dropout)"""
    model = Sequential()
    model.add(Input(shape=(32, 32, 3)))
    for i, f in enumerate(blocks):
        k = kernel_first if i == 0 else kernel           # розмір фільтра першого шару
        model.add(Conv2D(f, (k, k), padding='same', activation='relu'))
        if batchnorm: model.add(BatchNormalization())
        model.add(Conv2D(f, (kernel, kernel), padding='same', activation='relu'))
        if batchnorm: model.add(BatchNormalization())
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(conv_dropout))
    model.add(Flatten())
    for n in dense:                                       # класифікаційна частина
        model.add(Dense(n, activation='relu'))
        model.add(Dropout(dense_dropout))
    model.add(Dense(nb_classes, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

results = []
histories = {}

# базова модель — перший рядок таблиці
results.append({'експеримент': 'E0', 'що змінено': 'базова модель викладача',
                'епох': len(history.history['loss']), 'batch': batch_size,
                'train_acc': round(history.history['accuracy'][-1], 4),
                'val_acc': round(max(history.history['val_accuracy']), 4),
                'test_acc': round(scores[1], 4), 'test_loss': round(scores[0], 4),
                'параметрів': model.count_params(), 'час, с': round(base_time)})
histories['E0'] = history
best = {'name': 'E0', 'acc': scores[1], 'model': model}

def run_experiment(name, description, epochs=nb_epoch, batch_size=128, **cfg):
    print('=' * 80); print(name, '—', description); print('=' * 80)
    m = build_model(**cfg)
    early = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    start = time.time()
    h = m.fit(X_train, Y_train, batch_size=batch_size, epochs=epochs,
              validation_split=0.1, shuffle=True, callbacks=[early], verbose=2)
    t = time.time() - start
    loss, acc = m.evaluate(X_test, Y_test, verbose=0)
    results.append({'експеримент': name, 'що змінено': description,
                    'епох': len(h.history['loss']), 'batch': batch_size,
                    'train_acc': round(h.history['accuracy'][-1], 4),
                    'val_acc': round(max(h.history['val_accuracy']), 4),
                    'test_acc': round(acc, 4), 'test_loss': round(loss, 4),
                    'параметрів': m.count_params(), 'час, с': round(t)})
    histories[name] = h
    if acc > best['acc']:
        best.update(name=name, acc=acc, model=m)
    plot_history(h, f'{name}: {description}')
    print(f'{name}: точність на тестових даних {acc*100:.2f}%  (зупинено на епосі {len(h.history["loss"])})')

### 7.1. Кількість згорткових блоків: додано третій блок (128 карт ознак)

In [ ]:
run_experiment('E1', '+3-й згортковий блок (128)', blocks=(32, 64, 128))

### 7.2. Розмір згорткового фільтра: 5×5 у першому шарі

In [ ]:
run_experiment('E2', 'фільтр 5×5 у першому шарі', kernel_first=5)

### 7.3. Кількість карт ознак: удвічі більше (64 і 128)

In [ ]:
run_experiment('E3', 'більше карт ознак (64, 128)', blocks=(64, 128))

### 7.4. Шар регуляризації: BatchNormalization + Dropout 0.3

In [ ]:
run_experiment('E4', 'BatchNormalization + Dropout 0.3', batchnorm=True, conv_dropout=0.3)

### 7.5. Кількість повнозв'язних шарів: 512 + 256

In [ ]:
run_experiment('E5', '2 повнозв’язні шари (512, 256)', dense=(512, 256))

### 7.6. Кількість епох: 50 (з ранньою зупинкою)

In [ ]:
run_experiment('E6', '50 епох', epochs=50)

### 7.7. Розмір міні-вибірки: 64 і 256

In [ ]:
run_experiment('E7', 'batch_size = 64', batch_size=64)

In [ ]:
run_experiment('E8', 'batch_size = 256', batch_size=256)

### 7.8. Оптимізатор: SGD і RMSprop

In [ ]:
run_experiment('E9', 'оптимізатор SGD', optimizer='sgd')

In [ ]:
run_experiment('E10', 'оптимізатор RMSprop', optimizer='rmsprop')

### 7.9. Комбінація змін
3 згорткові блоки з більшою кількістю карт ознак, фільтр 5×5 у першому шарі, BatchNormalization, Dropout 0.3, adam, до 50 епох.

In [ ]:
run_experiment('E11', 'комбінація: 3 блоки (64,128,256), 5×5, BatchNorm, Dropout 0.3, 50 епох',
               blocks=(64, 128, 256), kernel_first=5, batchnorm=True, conv_dropout=0.3, epochs=50)

## Порівняння експериментів

In [ ]:
res = pd.DataFrame(results)
res.to_csv('lab4_experiments.csv', index=False)
pd.set_option('display.max_colwidth', None)
res.sort_values('test_acc', ascending=False)

In [ ]:
plt.figure(figsize=(13, 4))
plt.subplot(1, 2, 1)
order = res.sort_values('test_acc')
plt.barh(order['експеримент'] + ' ' + order['що змінено'].str[:35], order['test_acc'] * 100,
         color=['tab:green' if n == best['name'] else 'tab:blue' for n in order['експеримент']])
plt.xlabel('Точність на тестових даних, %'); plt.xlim(order['test_acc'].min() * 100 - 5, 100)
plt.title('Точність на тестових даних')
plt.subplot(1, 2, 2)
for name, h in histories.items():
    plt.plot(h.history['val_accuracy'], label=name, lw=2.5 if name == best['name'] else 1)
plt.xlabel('Epochs'); plt.ylabel('Validation accuracy'); plt.title('Точність на перевірці'); plt.legend(ncol=2, fontsize=8)
plt.tight_layout(); plt.show()
print('Найкраща модель:', best['name'], f"— {best['acc']*100:.2f}%")

## 8. Зберігаємо найкращу навчену нейрону мережу

In [ ]:
model = best['model']
model.summary()

model_json = model.to_json()
json_file = open("cifar10_model.json", "w")
json_file.write(model_json)
json_file.close()
model.save_weights("cifar10_model.weights.h5")   # Keras 3: ім'я має закінчуватися на .weights.h5
model.save("cifar10_best_model.keras")             # повна модель одним файлом

In [ ]:
!ls -la

In [ ]:
files.download("cifar10_model.json")
files.download("cifar10_model.weights.h5")
files.download("cifar10_best_model.keras")
files.download("lab4_experiments.csv")

## 9. Завантажуємо збережену мережу з файлу та застосовуємо її для розпізнавання

In [ ]:
json_file = open("cifar10_model.json", "r")
loaded_model_json = json_file.read()
json_file.close()
loaded_model = model_from_json(loaded_model_json)
loaded_model.load_weights("cifar10_model.weights.h5")
loaded_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

scores = loaded_model.evaluate(X_test, Y_test, verbose=0)
print("Точність завантаженої моделі на тестових даних: %.2f%%" % (scores[1]*100))

### Розпізнавання зображень з тестового набору

**Перегляд зображення з набору даних для тестування**

In [ ]:
index=22
plt.imshow(X_test[index])
plt.show()

**Перетворення тестового зображення**

In [ ]:
x = X_test[index]
x = np.expand_dims(x, axis=0)

**Запуск распізнавання**

In [ ]:
prediction = loaded_model.predict(x)

**Друкуємо результати розпізнавання**

In [ ]:
print(prediction)

**Перетворюємо результати з формату one hot encoding**

In [ ]:
prediction = np.argmax(prediction)
print(classes[prediction])

**Друкуємо вірну відповідь**

In [ ]:
print(classes[y_test[index][0]])

### Розпізнавання 20 зображень з тестового набору

In [ ]:
idx = np.random.RandomState(7).choice(len(X_test), 20, replace=False)
preds = loaded_model.predict(X_test[idx], verbose=0)
correct = 0
plt.figure(figsize=(15, 7))
for i, j in enumerate(idx):
    k, t = np.argmax(preds[i]), y_test[j][0]
    correct += int(k == t)
    plt.subplot(2, 10, i + 1); plt.imshow(X_test[j]); plt.axis('off')
    plt.title(f'{classes[k]}\n{preds[i][k]*100:.0f}%\n(вірно: {classes[t]})', fontsize=8,
              color='green' if k == t else 'red')
plt.suptitle(f'Тестова вибірка: вірно {correct} з 20'); plt.tight_layout(); plt.show()

## Розпізнаємо додаткові зображення

Завантажте 10–20 своїх фото (jpg/png), по одному об'єкту класу CIFAR-10 на фото.
Щоб програма сама рахувала вірні відповіді, назвіть файли з префіксом класу:

| префікс | клас | префікс | клас |
|---|---|---|---|
| `airplane_` | літак | `dog_` | собака |
| `automobile_` | автомобіль | `frog_` | жаба |
| `bird_` | птах | `horse_` | кінь |
| `cat_` | кіт | `ship_` | корабель |
| `deer_` | олень | `truck_` | вантажівка |

Наприклад: `cat_1.jpg`, `truck_2.png`. Найкраще — об'єкт великий і по центру.

In [ ]:
os.makedirs('my_images', exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, os.path.join('my_images', name))

Перевіряємо завантаження файлів

In [ ]:
image_paths = sorted(os.path.join('my_images', f) for f in os.listdir('my_images'))
print(len(image_paths), 'файлів:', [os.path.basename(p) for p in image_paths])

**Дивимось завантажений малюнок**

In [ ]:
img_path = image_paths[0]
img = load_img(img_path, target_size=(32, 32))
plt.imshow(img)
plt.show()

**Перетворюємо малюнок в масив для розпізнавання**

In [ ]:
x = img_to_array(img)
x /= 255
x = np.expand_dims(x, axis=0)

**Запускаємо розпізнавання**

In [ ]:
prediction = loaded_model.predict(x)
prediction = np.argmax(prediction)
print(classes[prediction])

### Розпізнавання всіх власних зображень
`load_img(target_size=(32, 32))` просто стискає фото до квадрата і спотворює пропорції, тому тут фото
спочатку обрізається до квадрата по центру (`ImageOps.fit`), а потім зменшується до 32×32.

In [ ]:
label_keys = {'airplane': 0, 'automobile': 1, 'bird': 2, 'cat': 3, 'deer': 4,
              'dog': 5, 'frog': 6, 'horse': 7, 'ship': 8, 'truck': 9}

def true_label(path):
    name = os.path.basename(path).lower()
    for key, idx in label_keys.items():
        if name.startswith(key):
            return idx
    return None

def prepare_image(path):
    img = Image.open(path).convert('RGB')
    img = ImageOps.fit(img, (32, 32), Image.LANCZOS)   # обрізка до квадрата по центру + зменшення
    return np.array(img).astype('float32') / 255

imgs = np.array([prepare_image(p) for p in image_paths])
preds = loaded_model.predict(imgs, verbose=0)

cols = 5; rows_n = int(np.ceil(len(image_paths) / cols))
plt.figure(figsize=(cols * 2.8, rows_n * 3.2))
rows, correct, known = [], 0, 0
for i, p in enumerate(image_paths):
    order = np.argsort(preds[i])[::-1]
    k, t = order[0], true_label(p)
    ok = None if t is None else bool(k == t)
    if t is not None:
        known += 1; correct += int(ok)
    plt.subplot(rows_n, cols, i + 1)
    plt.imshow(Image.open(p).convert('RGB')); plt.axis('off')
    plt.title(f'{classes[k]} {preds[i][k]*100:.0f}%', fontsize=10,
              color='black' if ok is None else ('green' if ok else 'red'))
    rows.append({'файл': os.path.basename(p),
                 'справжній клас': classes[t] if t is not None else '?',
                 'передбачено': classes[k], 'впевненість, %': round(float(preds[i][k]) * 100, 1),
                 'топ-3': ', '.join(f'{classes[c]} {preds[i][c]*100:.0f}%' for c in order[:3]),
                 'вірно': ok})
plt.tight_layout(); plt.show()
if known:
    print(f'Власні зображення: вірно {correct} з {known} ({correct/known*100:.0f}%)')
pd.DataFrame(rows)

In [ ]:
# Як бачить зображення мережа (32×32)
plt.figure(figsize=(cols * 2.2, rows_n * 2.2))
for i in range(len(imgs)):
    plt.subplot(rows_n, cols, i + 1); plt.imshow(imgs[i]); plt.axis('off')
plt.suptitle('Зображення після зменшення до 32×32'); plt.tight_layout(); plt.show()

## Архів із зображеннями

In [ ]:
!zip -q -r lab4_images.zip my_images
files.download('lab4_images.zip')

## Висновки

*(заповніть після запуску)*

- Базова модель: точність на тестових даних ...
- Найкраща комбінація гіперпараметрів: ..., точність ...
- Що найбільше вплинуло на точність: ...
- Ознаки перенавчання на графіках: ...
- Тестова вибірка: вірно ... з 20. Власні зображення: вірно ... з ...